In [ ]:
%load_ext dotenv
%dotenv config.env

In [ ]:
%load_ext autoreload 
%autoreload 2

In [ ]:
import random
import h5py
import importlib
import subprocess
import uuid
import trackpy as tp
import numpy as np
import pandas as pd
import os
import pickle
from matplotlib import pyplot as plt
from pathlib import Path
from ipyfilechooser import FileChooser
from utils.conditions_to_dict import conditions_to_df, get_fovs
from ipywidgets import interactive, fixed
from utils.set_widgets import filter_data, h5_w_tracking_params, manual_reg_widgets, fov_selector

from utils.conditions_to_dict import conditions_to_df, get_fovs
from utils.logging import setup_logger, log_and_print, save_as_pickle
from utils.set_widgets import *
from utils.plot import *
from utils.tiff_writer import download_img_allas, show_mask
from utils.locs_trackpy import get_locs_trackpy, h5_get_locs_preview
from utils.track_trackpy import track_py, plot_tracks, locs_preview_contours
from skimage import measure
from utils.csv_writer import show_df_style
from io import BytesIO
from utils.dnaK_proj import get_intensities
from shapely import points, distance

In [ ]:
# Select the aoutput directory to save plots, logs and data 

path = Path(os.getenv('SMPP_INPUT_DIR'))
output_for_data = FileChooser(path)
output_for_data.show_only_dirs = True
display(output_for_data)


In [ ]:
# setup logging

analysis_id = str(uuid.uuid1())[:8]
git_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode('ascii').strip()

logger = setup_logger(output_for_data, analysis_id)
log_and_print(f'output directory: {output_for_data.selected_path}')
log_and_print(f'analysis_id_short:{analysis_id}')
log_and_print(f'git_hash:{git_hash}')


In [ ]:
# Click 'Play' to search across the database and choose the experiment which you'd like to analyse.
# This script will only work with the experiments performed on PyME gui after 27.02.2025

# Only one experiment entry is allowed to be processed.
# Use the 'clear' function in the 'date' field to remove any unwanted entry.
# It is ok to use a comma and space to separate the row numbers in the 'exclude' and 'include' fields.
# Keep all the fields intact to view the entire database (though this may become inconvenient soon)

database = pd.read_csv(os.getenv('SMPP_DATABASE'), sep = '\t', dtype = 'str')
fields_list, fld_widgets_list, box = filter_data(database)
display(box)


In [ ]:
# Run this cell when ready with the selection above,u
# check if the experiment you choose is the one you want.

experiment = conditions_to_df(database, fld_widgets_list, fields_list, 'Experiments')
experiment


In [ ]:
# Run this cell to see selected imaging files

exp_id = experiment['uuid_experiment'].values[0]
files, masks = get_fovs(database, exp_id)

log_and_print('\nSelected files:')
show_df_style(files)
log_and_print('\n\nSelected masks:')
show_df_style(masks)
log_and_print('')


In [ ]:
# Select the FOVs you'd like to process (must have a mask associated, see 'masks:' field)

box, output, checked_ids = fov_all_selector(files, masks)
log_and_print('')
display(box, output)
log_and_print('')


In [ ]:
# Show selected FOVs

files_selected = files[files['uuid_fov'].isin(checked_ids)]
masks_selected = masks[masks['uuid_fov'].isin(checked_ids)]

log_and_print(f"\n\n{files_selected.shape[0]} files selected, {masks_selected.shape[0]} masks\n")
show_df_style(files_selected)
log_and_print('')


In [ ]:
# Get the tracking data from Allas (only the tracking channel for each FOV will be shown)
# (might take few min)

master_im_data = []
master_im_data_2ch = []
master_fluos = []
master_fluos_2ch = []

for file in files_selected['uuid_file'].unique():
    path = eval(files.loc[files['uuid_file']==file, 'path'].item())
    h5 = h5py.File(BytesIO(download_img_allas(path[0], path[1])), 'r')
    log_and_print(f"{path[1]} downloaded")
    
    if h5['ImageData'].shape[0] > 10**4:

        
        try:
            fov_id = files.loc[files['uuid_file']==file, 'uuid_fov'].item()
            p_mask = masks.loc[masks['uuid_fov']==fov_id, 'path'].item()
            mask_uuid = masks.loc[masks['uuid_fov']==fov_id, 'uuid_analysis'].item()
            
            master_im_data.append((file, path, p_mask, mask_uuid))
            master_fluos.append(h5['ImageData'])

            m_fname = p_mask.split('/')[-1]
            m_dir   = '/'.join(p_mask.split('/')[:-1])
            m = show_mask(m_dir, m_fname, imshow=False)
            proj = np.mean(h5['ImageData'][0:42,:,:], axis=0)

            fig, axs = plt.subplots(1, 2)
            axs[0].imshow(proj)
            axs[1].imshow(m)
            plt.show()
            plt.close()
            
            log_and_print(f"FOV: {fov_id}")
            log_and_print(f"{path[1]} added to fluos (tracking)")
            print('\n================================================\n\n')
            
        except ValueError:
            log_and_print(f'can\'t find mask for file {path[1]}')

    elif h5['MetaData'].attrs['LEDpower']==0: # not widefield
           
        try:
            fov_id = files.loc[files['uuid_file']==file, 'uuid_fov'].item()
            p_mask = masks.loc[masks['uuid_fov']==fov_id, 'path'].item()
            mask_uuid = masks.loc[masks['uuid_fov']==fov_id, 'uuid_analysis'].item()
            
            master_im_data_2ch.append((file, path, p_mask, mask_uuid))
            master_fluos_2ch.append(h5['ImageData'])

            m_fname = p_mask.split('/')[-1]
            m_dir   = '/'.join(p_mask.split('/')[:-1])
            m = show_mask(m_dir, m_fname, imshow=False)
            proj = np.mean(h5['ImageData'][0:42,:,:], axis=0)

            fig, axs = plt.subplots(1, 2)
            axs[0].imshow(proj)
            axs[1].imshow(m)
            plt.show()
            plt.close()

            log_and_print(f"FOV: {fov_id}")
            log_and_print(f"{path[1]} added to 2nd_channel ")
            print('\n================================================\n\n')
            
        except ValueError:
            log_and_print(f'can\'t find mask for file {path[1]}')
    
    else: 
        plt.imshow(np.mean(h5['ImageData'][0:42,:,:], axis=0))
        plt.show()
        plt.close()

        fov_id = files.loc[files['uuid_file']==file, 'uuid_fov'].item()
        log_and_print(f"FOV: {fov_id}")
        log_and_print(f"{path[1]} is brightfield")
        print('\n================================================\n\n')
        

print('All files are ready')

In [ ]:
# This cell will generate the projections

projections = []
images = []

for file, info in zip(master_fluos_2ch, master_im_data_2ch):
    print(file.shape)
    proj = np.mean(file[1:,:,:], axis = 1)
    plt.imshow(proj)
    plt.show()
    proj = np.mean(proj, axis = 1)
    plt.plot(proj)
    plt.show()

    f_brightest = np.argmax(proj)
    brightest = file[f_brightest]
    print('brightest frame ' + str(f_brightest))
    plt.imshow(brightest)
    plt.show()
    
    projections.append([brightest, info[2], info[0], info[3]])
    images.append(brightest)


## Get intensities for each cell in the 2nd fluorescence channel

In [ ]:
combined_per_cell = []

for i in range(len(projections)):
    pcell =  get_intensities(*projections[i], verbose = False)
    combined_per_cell.append(pcell)

combined_per_cell = pd.concat(combined_per_cell)

In [ ]:
# Dump the data

directory = 'path_here'
filename = "filename_here"
combined_per_cell.to_csv(f'{directory}/{filename}_{exp_id[:8]}.csv')
print('Data saved')
